# KO CBF Generation
Generate a number of keep-out (KO) regions centered on and around the path with feasible velocities and accelerations which will be used to train the controller while still producing a guaranteed safe output at all times.

In [1]:
import pathlib
import h5py

import numpy as np

import itertools

In [2]:
episodes_train_dir = pathlib.Path("../data/episodes/train")
episodes_test_dir = pathlib.Path("../data/episodes/test")

In [3]:
# load all the episodes from file into memory


def load_from_h5(h5_file_dir):
    episodes = []
    files = []
    for file in h5_file_dir.iterdir():
        if file.name == ".gitkeep":
            continue  # don't try to load this one
        episode = dict()
        with h5py.File(file, "r") as f:
            f.visititems(lambda name, obj: episode.update({name: np.asarray(obj)}))
            episode.update(f.attrs)
        episodes.append(episode)
        files.append(file.name)
    return episodes, files


train_episodes, train_files = load_from_h5(episodes_train_dir)
test_episodes, test_files = load_from_h5(episodes_test_dir)

train_episodes[0]

{'ddx_traj': array([ 0.02141311,  0.02131748,  0.02122186,  0.02112623,  0.0210306 ,
         0.02093497,  0.02083934,  0.02074371,  0.02064808,  0.02055246,
         0.02045683,  0.0203612 ,  0.02026557,  0.02016994,  0.02007431,
         0.01997869,  0.01988306,  0.01978743,  0.0196918 ,  0.01959617,
         0.01950054,  0.01940491,  0.01930929,  0.01921366,  0.01911803,
         0.0190224 ,  0.01892677,  0.01883114,  0.01873552,  0.01863989,
         0.01854426,  0.01844863,  0.018353  ,  0.01825737,  0.01816174,
         0.01806612,  0.01797049,  0.01787486,  0.01777923,  0.0176836 ,
         0.01758797,  0.01749235,  0.01739672,  0.01730109,  0.01720546,
         0.01710983,  0.0170142 ,  0.01691857,  0.01682295,  0.01672732,
         0.01663169,  0.01653606,  0.01644043,  0.0163448 ,  0.01624918,
         0.01615355,  0.01605792,  0.01596229,  0.01586666,  0.01577103,
         0.0156754 ,  0.01557978,  0.01548415,  0.01538852,  0.01529289,
         0.01519726,  0.01510163,  0.01

In [ ]:
num_ko_regions = 10  # number of KO regions to generate for each trajectory

# standard deviations (zero mean unless tuple)
dist_ko_offsets = 0.5 * np.diag([1.0, 1.0])
dist_vel_x = 0.5
dist_vel_y = dist_vel_x
dist_accel_x = 0.1
dist_accel_y = dist_accel_x

dist_radius = 1.0, 0.4  # mean, deviation
dist_vel_radius = 0.3
dist_accel_radius = 0.1

r = np.random.default_rng(42)  # for reproducibility

In [5]:
keep_out_train_dir = pathlib.Path("../data/keep_out/train")
keep_out_test_dir = pathlib.Path("../data/keep_out/test")


# empty these repositories so we have no issues when writing the episodes that
# were generated in the preceding section
for file in itertools.chain(keep_out_train_dir.iterdir(), keep_out_test_dir.iterdir()):
    if file.name == ".gitkeep":
        continue  # don't delete this file
    file.unlink()

num_train_episodes = len(train_episodes)
num_test_episodes = len(test_episodes)

for i in range(num_train_episodes + num_test_episodes):
    episode = (
        train_episodes[i]
        if i < num_train_episodes
        else test_episodes[i - num_train_episodes]
    )

    # first generated all the offsets and properties of the keep-out regions
    ko_traj_offsets = r.multivariate_normal(
        np.zeros(2), dist_ko_offsets, (num_ko_regions)
    )
    ko_vel_x = r.normal(0.0, dist_vel_x, (num_ko_regions))
    ko_vel_y = r.normal(0.0, dist_vel_y, (num_ko_regions))
    ko_accel_x = r.normal(0.0, dist_accel_x, (num_ko_regions))
    ko_accel_y = r.normal(0.0, dist_accel_y, (num_ko_regions))

    ko_radius = r.normal(dist_radius[0], dist_radius[1], (num_ko_regions))
    ko_vel_radius = r.normal(0.0, dist_vel_radius, (num_ko_regions))
    ko_accel_radius = r.normal(0.0, dist_accel_radius, (num_ko_regions))

    # now just need to find random points to place these keep-out regions along
    # the trajectory of the system (with appropriate offsets)

    # NOTE: this is highly inefficient but we just need to run it "once" during
    # dataset generation so it doesn't matter

    unique_rand_idxs = []
    while len(unique_rand_idxs) < num_ko_regions:
        idx = r.integers(0, len(episode["t_traj"]))
        if idx not in unique_rand_idxs:
            unique_rand_idxs.append(idx)

    ko_x = ko_traj_offsets[:, 0] + episode["x_traj"][unique_rand_idxs]
    ko_y = ko_traj_offsets[:, 1] + episode["y_traj"][unique_rand_idxs]

    # stores the results in the corresponding file

    file = (
        keep_out_train_dir.joinpath(train_files[i])
        if i < num_train_episodes
        else keep_out_test_dir.joinpath(test_files[i - num_train_episodes])
    )
    with h5py.File(file, "w") as f:
        f.create_dataset("ko_x", data=ko_x)
        f.create_dataset("ko_y", data=ko_y)
        f.create_dataset("ko_vel_x", data=ko_vel_x)
        f.create_dataset("ko_vel_y", data=ko_vel_y)
        f.create_dataset("ko_accel_x", data=ko_accel_x)
        f.create_dataset("ko_accel_y", data=ko_accel_y)
        f.create_dataset("ko_radius", data=ko_radius)
        f.create_dataset("ko_vel_radius", data=ko_vel_radius)
        f.create_dataset("ko_accel_radius", data=ko_accel_radius)